In [1]:
# notebook: 07_macro_cmvts_final_validation.ipynb
# ============================================================================
# CMVTS Extension — macro-CMVTS assembly & final validation  (Section 5.2/5.4/5.5)
# ----------------------------------------------------------------------------
# Scope decision (branch 1): C1 is EXCLUDED from this validation. C1 is defined
# on the TARGET proxy distribution, which is exactly the data used to build the
# outcome Y. Including C1 would re-introduce circularity. We therefore validate
# the MACRO components only:  macro-CMVTS = w2*C2 + w3*C3  (w2+w3=1).
# C1 is discussed separately in the paper as "not independently testable against
# an outcome derived from the same target distribution."
#
# Inputs (from the CORRECTED notebook 05):
#   cmvts_C2C3_latest.csv : columns C2, C3, Y  (Y = Branch-2 penetration JSD)
#   cmvts_C2C3_2021.csv   : columns C2, C3     (2021 vintage, for design-C)
#
# Produces:
#   5.2  macro-CMVTS vs outcome (headline validation, circularity-free)
#   5.4  weight re-derivation: which (w2,w3) maximizes predictive power
#   5.5  design-C: 2021 vs latest vintage macro-CMVTS
# ============================================================================

import numpy as np
import pandas as pd
from scipy import stats

# ----------------------------------------------------------------------------
# 0. LOAD corrected C2/C3 + outcome
# ----------------------------------------------------------------------------
lat = pd.read_csv("cmvts_C2C3_latest.csv", index_col=0)   # C2, C3, Y
v21 = pd.read_csv("cmvts_C2C3_2021.csv",  index_col=0)     # C2, C3
assert {"C2","C3","Y"}.issubset(lat.columns), "notebook 05 output missing columns"

ORDER = ["Indonesia","Thailand","Viet Nam","Philippines","Bangladesh",
         "Cambodia","Nepal","Pakistan","Lao PDR"]
lat = lat.reindex(ORDER); v21 = v21.reindex(ORDER)
Y = lat["Y"]

def corr(x, y):
    c = x.notna() & y.notna()
    r,p   = stats.pearsonr(x[c], y[c])
    rs,ps = stats.spearmanr(x[c], y[c])
    return r,p,rs,ps

def show(x, tag, y=Y):
    r,p,rs,ps = corr(x,y)
    print(f"  [{tag:22s}] Pearson {r:+.3f}(p={p:.3f}) | Spearman {rs:+.3f}(p={ps:.3f})")
    return rs

# higher macro-CMVTS = more similar -> should predict LOWER divergence (negative r)

# ============================================================================
# 5.2 — Headline validation: macro-CMVTS vs outcome (paper default weights)
# ============================================================================
# Original paper macro split is C2:C3 = 0.3:0.3 -> equal within macro = 0.5:0.5.
print("="*70); print("5.2 — macro-CMVTS vs realized divergence (circularity-free)")
print("="*70)
print("Circularity guarantee: predictor uses only C2,C3 (macro structure);")
print("outcome Y uses only target card-activity penetration. Disjoint inputs.\n")

def macro_cmvts(df, w2, w3):
    return w2*df["C2"] + w3*df["C3"]

for (w2,w3,tag) in [(0.5,0.5,"equal (paper macro)"),
                    (1.0,0.0,"C2 only"),
                    (0.0,1.0,"C3 only")]:
    show(macro_cmvts(lat,w2,w3), f"w2={w2},w3={w3} {tag}")

headline = macro_cmvts(lat,0.5,0.5)
print("\nPer-country (equal-weight macro-CMVTS):")
print(pd.DataFrame({"macroCMVTS":headline.round(4),"Y":Y.round(4)})
      .sort_values("Y").to_string())

# ============================================================================
# 5.4 — Weight re-derivation: grid over (w2,w3), maximize |Spearman| with Y
# ============================================================================
print("\n"+"="*70); print("5.4 — Weight re-derivation (macro split)"); print("="*70)
grid=[]
for w2 in np.round(np.arange(0,1.0001,0.05),2):
    w3=round(1-w2,2)
    rs=stats.spearmanr(macro_cmvts(lat,w2,w3)[Y.index], Y)[0]
    r =stats.pearsonr(macro_cmvts(lat,w2,w3)[Y.index], Y)[0]
    grid.append({"w2":w2,"w3":w3,"pearson":round(r,3),"spearman":round(rs,3)})
G=pd.DataFrame(grid)
best_s=G.loc[G["spearman"].idxmin()]   # most negative
best_p=G.loc[G["pearson"].idxmin()]
print("Grid (every 0.05):")
print(G.to_string(index=False))
print(f"\nStrongest Spearman: w2={best_s.w2}, w3={best_s.w3} -> {best_s.spearman}")
print(f"Strongest Pearson : w2={best_p.w2}, w3={best_p.w3} -> {best_p.pearson}")
print("Compare to paper's implied equal macro split (0.5/0.5). If the optimum is")
print("far from equal, that is evidence the weights should be data-derived, not")
print("the arbitrary 0.4/0.3/0.3 the original paper acknowledged as a limitation.")

# robustness: is the verdict-ranking stable across ALL macro weights?
sp_range=(G["spearman"].min(), G["spearman"].max())
print(f"\nSpearman range across all macro weights: {sp_range[0]:+.3f} to {sp_range[1]:+.3f}")
print("Narrow & always-negative => predictive validity is weight-insensitive.")

# ============================================================================
# 5.5 — Design-C: 2021 vs latest vintage macro-CMVTS (time sensitivity)
# ============================================================================
print("\n"+"="*70); print("5.5 — Design-C vintage sensitivity"); print("="*70)
mc_lat = macro_cmvts(lat,0.5,0.5)
mc_21  = 0.5*v21["C2"] + 0.5*v21["C3"]
comp = pd.DataFrame({"macroCMVTS_2021":mc_21.round(4),
                     "macroCMVTS_latest":mc_lat.round(4),
                     "delta":(mc_lat-mc_21).round(4),
                     "Y":Y.round(4)}).reindex(ORDER)
print(comp.to_string())
print("\nCorrelation with Y at each vintage:")
show(mc_21, "macro-CMVTS 2021")
show(mc_lat,"macro-CMVTS latest")
print("\nMean |vintage shift|: %.4f" % (mc_lat-mc_21).abs().mean())
print("Interpretation: small, sign-stable shifts => verdict robust to the 2022/2024")
print("time gap between the CB source and Findex targets (design-choice C).")

# ============================================================================
# SUMMARY for the manuscript
# ============================================================================
print("\n"+"="*70); print("MANUSCRIPT SUMMARY"); print("="*70)
r,p,rs,ps = corr(headline, Y)
print(f"Headline (5.2): macro-CMVTS(equal) vs divergence  Spearman {rs:+.3f} (p={ps:.3f})")
print(f"Best weights (5.4): w2={best_s.w2}/w3={best_s.w3}  Spearman {best_s.spearman}")
print(f"Vintage stability (5.5): mean|shift| {(mc_lat-mc_21).abs().mean():.4f}, sign stable")
print("\nAll three sections rest on a circularity-free predictor (C2+C3 macro only).")
print("C1 is excluded by design and discussed separately (target-distribution based).")

# save the headline series for figures
pd.DataFrame({"macroCMVTS":headline,"Y":Y}).to_csv("macro_cmvts_headline.csv")
print("\nSaved macro_cmvts_headline.csv")

5.2 — macro-CMVTS vs realized divergence (circularity-free)
Circularity guarantee: predictor uses only C2,C3 (macro structure);
outcome Y uses only target card-activity penetration. Disjoint inputs.

  [w2=0.5,w3=0.5 equal (paper macro)] Pearson -0.775(p=0.014) | Spearman -0.817(p=0.007)
  [w2=1.0,w3=0.0 C2 only ] Pearson -0.764(p=0.017) | Spearman -0.817(p=0.007)
  [w2=0.0,w3=1.0 C3 only ] Pearson -0.787(p=0.012) | Spearman -0.883(p=0.002)

Per-country (equal-weight macro-CMVTS):
             macroCMVTS       Y
Viet Nam         0.6842  0.0000
Thailand         0.7762  0.0092
Nepal            0.5874  0.0332
Indonesia        0.6734  0.0708
Lao PDR          0.5434  0.0753
Cambodia         0.6034  0.0810
Philippines      0.5650  0.1809
Bangladesh       0.5561  0.2110
Pakistan         0.4412  0.2192

5.4 — Weight re-derivation (macro split)
Grid (every 0.05):
  w2   w3  pearson  spearman
0.00 1.00   -0.787    -0.883
0.05 0.95   -0.786    -0.883
0.10 0.90   -0.785    -0.833
0.15 0.85   -0.78